 # KuroSiwo Dataset: Exploratory Data Analysis



 KuroSiwo is a large-scale, multi-temporal SAR dataset for flood detection with 43 real flood events, 1.73M catalogue entries, and ~1.6M exported patches. This notebook explores the catalogue to answer:



 1. **What's in the catalogue?** Schema, product types, quality flags

 2. **How are floods labelled?** Flood percentage distribution, class imbalance

 3. **How are events organised?** Per-event patch counts and coverage

 4. **Is there enough metadata per flood case?** Dates, bounding boxes, flood extent

In [ ]:
%reset -f
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import subprocess
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


In [ ]:
# ============================================================================
# SETUP: Data Download & Configuration
# ============================================================================
# NOTE: Original catalogue URL (Dropbox):
# "https://www.dropbox.com/scl/fi/wu6nvj73cz4h7k3gxpzx6/catalogue.gpkg"
# ?rlkey=hsij2o0k60r2n0z6z4d2ngww9&st=0zjqhzgx&dl=1

from pathlib import Path
import shutil

catalogue_path = Path("../../assets/ks_catalogue.gpkg")

def is_lfs_pointer(path: Path) -> bool:
    try:
        with open(path, 'r', encoding='utf8') as fh:
            first = fh.readline()
        return first.startswith('version https://git-lfs.github.com/spec')
    except Exception:
        return False

def run_cmd(cmd, cwd=None):
    try:
        subprocess.run(cmd, check=True, cwd=cwd)
        return True
    except FileNotFoundError:
        return False
    except subprocess.CalledProcessError:
        return False

# Ensure the asset is present and not an LFS pointer; try to fetch if needed
if catalogue_path.exists():
    if is_lfs_pointer(catalogue_path):
        try:
            subprocess.run(['git', 'lfs', 'pull'], check=True)
        except subprocess.CalledProcessError:
            raise RuntimeError(
                "`git lfs pull` failed; probably git lfs is not installed. "
                "Try: git lfs install, then retry."
            )
        else:
            print(f"Catalogue ready at {catalogue_path}")
else:
    raise Exception(
        f"Catalogue not found at {catalogue_path}, and no way to fetch it (git lfs not available)"
    )

gdf = gpd.read_file(str(catalogue_path))

In [ ]:
print(f"Loaded: {gdf.shape[0]:,} rows × {gdf.shape[1]} columns")
print(f"\nAll columns:\n{gdf.columns.tolist()}")
gdf.crs

In [ ]:
pd.set_option('display.max_colwidth', None)
gdf.sample(3)

 ## 1. Dataset Overview



 What is the top-level shape of this dataset?

In [ ]:
# How many events, patches and what time span does the catalogue cover?
print(f"Total catalogue rows:          {len(gdf):,}")
print(f"Unique spatial patches:        {gdf['grid_id'].nunique():,}")
print(f"Unique flood events (actid):   {gdf['actid'].nunique()}")
print(f"Date range:                    {gdf['flood_date'].min().date()} → {gdf['flood_date'].max().date()}")
print(f"Source Date range:             {gdf['source_date'].min().date()} → {gdf['source_date'].max().date()}")

mask_source_after_flood = pd.to_datetime(gdf['source_date']) > pd.to_datetime(gdf['flood_date'])
n_source_after_flood = mask_source_after_flood.sum()
print(f"Rows with source_date > flood_date: {n_source_after_flood:,} ({100*n_source_after_flood/len(gdf):.2f}%)")

In [ ]:
# What product types are available? crank=1 → GRD (amplitude), crank=2 → SLC (complex)
for crank, label in {1: 'GRD (amplitude)', 2: 'SLC (complex)'}.items():
    n = (gdf['crank'] == crank).sum()
    print(f"  crank={crank}  {label:20s}  {n:>10,}  ({100*n/len(gdf):.1f}%)")

In [ ]:
# What is the temporal role of each patch? master=True → flood-time, master=False → pre-flood baseline
n_flood    = (gdf['master'] == True).sum()
n_preflood = (gdf['master'] == False).sum()
print(f"  master=True  (flood-time):       {n_flood:>10,}  ({100*n_flood/len(gdf):.1f}%)")
print(f"  master=False (pre-flood baseline):{n_preflood:>10,}  ({100*n_preflood/len(gdf):.1f}%)")

In [ ]:
# How many patches are actually exported to disk and pass quality validation?
n_exported = (gdf['exported'] == True).sum()
n_valid    = (gdf['gvalid']   == True).sum()
print(f"  Exported (on disk):       {n_exported:>10,}  ({100*n_exported/len(gdf):.1f}%)")
print(f"  Quality valid (gvalid):   {n_valid:>10,}  ({100*n_valid/len(gdf):.1f}%)")

 ## 2. Flood Labels & Class Distribution



 Each patch carries a `pflood` value: the percentage of the 256×256 pixel tile covered by flood water. How is this distributed?

In [ ]:
# Flood label statistics — flood-time patches only (master=True, GRD product)
flood_patches = gdf[(gdf['master'] == True) & (gdf['crank'] == 1)]
s = flood_patches['pflood']
print(f"Flood-time patches (GRD):  {len(flood_patches):,}")
print(f"pflood range:              {s.min():.1f}% – {s.max():.1f}%")
print(f"Mean / Median:             {s.mean():.2f}% / {s.median():.2f}%")

In [ ]:
# Class breakdown — among patches that HAVE a flood label (pflood is not NaN)
s_labeled = s.dropna()
n_labeled = len(s_labeled)
n_unlabeled = len(s) - n_labeled
print(f"Flood-time patches with a pflood label:    {n_labeled:>8,}  ({100*n_labeled/len(s):.1f}%)")
print(f"Flood-time patches without label (NaN):    {n_unlabeled:>8,}  ({100*n_unlabeled/len(s):.1f}%)")
print()
no_flood  = (s_labeled == 0).sum()
low       = ((s_labeled > 0)  & (s_labeled <= 5)).sum()
mid       = ((s_labeled > 5)  & (s_labeled <= 50)).sum()
high      = (s_labeled > 50).sum()
for label, n in [('No flood (0%)', no_flood), ('Low (0–5%)', low), ('Significant (5–50%)', mid), ('Dominant (>50%)', high)]:
    print(f"  {label:25s}  {n:>8,}  ({100*n/n_labeled:.1f}% of labeled)")

In [ ]:
# Visualise the pflood distribution and class imbalance (labeled patches only)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(s_labeled, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(s_labeled.median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: {s_labeled.median():.1f}%')
ax1.set_xlabel('Flood coverage per patch (%)')
ax1.set_ylabel('Number of patches')
ax1.set_title('Distribution of pflood (labeled patches)')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

counts = [no_flood, low, mid, high]
labels = ['No flood\n(0%)', 'Low\n(0–5%)', 'Significant\n(5–50%)', 'Dominant\n(>50%)']
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#c0392b']
ax2.pie(counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Class distribution (labeled flood-time patches)')

plt.tight_layout()
plt.show()

 ## 3. Event-Level Breakdown



 How many patches does each flood event contribute, and do all events have both flood-time and pre-flood acquisitions?

In [ ]:
# Build a per-event summary (GRD patches only, crank=1)
records = []
for actid in sorted(gdf['actid'].unique()):
    ev = gdf[gdf['actid'] == actid]
    grd = ev[ev['crank'] == 1]
    records.append({
        'event_id': actid,
        'flood_date': grd[grd['master'] == True]['flood_date'].iloc[0].date(),
        'flood_patches': (grd['master'] == True).sum(),
        'preflood_patches': (grd['master'] == False).sum(),
        'valid_exported': grd[(grd['exported'] == True) & (grd['gvalid'] == True)].shape[0],
        'labeled_flood': grd[(grd['master'] == True) & (grd['pflood'] > 0)].shape[0],
    })
event_meta_df = pd.DataFrame(records)
print(f"Events with pre-flood baseline: {(event_meta_df['preflood_patches'] > 0).sum()} / {len(event_meta_df)}")
print(f"Total flood patches:            {event_meta_df['flood_patches'].sum():,}")
print(f"Total pre-flood patches:        {event_meta_df['preflood_patches'].sum():,}")
print(f"Total valid+exported patches:   {event_meta_df['valid_exported'].sum():,}")
print(f"Patches with flood label >0%:   {event_meta_df['labeled_flood'].sum():,}")

In [ ]:
# Which events are the largest? Top 15 by valid exported patch count
top15 = event_meta_df.nlargest(15, 'valid_exported')
print("Top 15 events by valid exported patches:")
print(top15[['event_id', 'flood_date', 'flood_patches', 'preflood_patches', 'valid_exported', 'labeled_flood']].to_string(index=False))

In [ ]:
# Are labeled patches proportional to event size? (sanity check)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

top15_plot = event_meta_df.nlargest(15, 'valid_exported')
ax1.barh(range(len(top15_plot)), top15_plot['valid_exported'], color='steelblue', alpha=0.7)
ax1.set_yticks(range(len(top15_plot)))
ax1.set_yticklabels([f"Event {e}" for e in top15_plot['event_id']], fontsize=9)
ax1.set_xlabel('Valid exported patches')
ax1.set_title('Top 15 events by patch count')
ax1.grid(axis='x', alpha=0.3)

ax2.scatter(event_meta_df['valid_exported'], event_meta_df['labeled_flood'],
            s=60, alpha=0.7, color='coral', edgecolors='black', linewidth=0.4)
max_v = event_meta_df['valid_exported'].max()
ax2.plot([0, max_v], [0, max_v], 'k--', alpha=0.3, linewidth=1, label='1:1 line')
ax2.set_xlabel('Valid exported patches')
ax2.set_ylabel('Patches with flood label (pflood > 0%)')
ax2.set_title('Labeled coverage per event')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

 ## 4. Flood Case Metadata



 To run model equivalents we need per-case: `flood_case`, `date_start`, `date_end`, and `bounding_box(lat_min, lat_max, lon_min, lon_max)`. Useful extras are `max_flood_extent_km²` and `date_of_max_flood_extent`. Do we have these?

In [ ]:
# Reproject once for geographic bounds and once for area-accurate extent calculations
AREA_CRS = 'EPSG:6933'  # Equal-area projection for km² area estimates
EXTENT_CRANK = 1        # Use GRD patches for extent consistency
EXTENT_PRODUCT = {1: 'GRD', 2: 'SLC'}[EXTENT_CRANK]

gdf_wgs84 = gdf.to_crs('EPSG:4326')
gdf_area = gdf.to_crs(AREA_CRS).copy()
gdf_area['patch_area_km2'] = gdf_area.geometry.area / 1_000_000

case_records = []
for actid in sorted(gdf['actid'].unique()):
    ev_orig = gdf[gdf['actid'] == actid]
    ev_geo = gdf_wgs84[gdf_wgs84['actid'] == actid]
    ev_area = gdf_area[gdf_area['actid'] == actid]

    flood_rows = ev_orig[ev_orig['master'] == True]
    preflood_rows = ev_orig[ev_orig['master'] == False]

    lon_min, lat_min, lon_max, lat_max = ev_geo.total_bounds

    flooded = ev_area[
        (ev_area['master'] == True)
        & (ev_area['crank'] == EXTENT_CRANK)
        & (ev_area['pflood'].notna())
        & (ev_area['pflood'] > 0)
    ]
    extent_km2 = (flooded['patch_area_km2'] * flooded['pflood'] / 100).sum()

    case_records.append({
        'flood_case': f"KuroSiwo_{actid:03d}",
        'date_start': preflood_rows['flood_date'].iloc[0].date() if len(preflood_rows) else None,
        'date_end': flood_rows['flood_date'].iloc[0].date() if len(flood_rows) else None,
        'lat_min': round(lat_min, 4), 'lat_max': round(lat_max, 4),
        'lon_min': round(lon_min, 4), 'lon_max': round(lon_max, 4),
        'max_flood_extent_km2': round(extent_km2, 1),
        'date_of_max_flood_extent': flood_rows['flood_date'].iloc[0].date() if len(flood_rows) else None,
    })

flood_case_df = pd.DataFrame(case_records)
print(f"Extracted metadata for {len(flood_case_df)} flood cases.")
print(f"Extent method: sum(pflood_fraction × per-patch area in {AREA_CRS}, {EXTENT_PRODUCT} only)")

In [ ]:
# Sample the metadata table — do all required fields look correct?
print(flood_case_df[['flood_case', 'date_start', 'date_end', 'lat_min', 'lat_max', 'lon_min', 'lon_max']].to_string(index=False))

In [ ]:
# When did flood events occur? (temporal distribution by year)
years = pd.to_datetime(flood_case_df['date_end']).dt.year.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(years.index, years.values, color='steelblue', alpha=0.8, edgecolor='black')
ax.set_xlabel('Year')
ax.set_ylabel('Number of flood cases')
ax.set_title('Flood cases by year')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# How large are the bounding boxes? (degrees)
lat_span = flood_case_df['lat_max'] - flood_case_df['lat_min']
lon_span = flood_case_df['lon_max'] - flood_case_df['lon_min']
print(f"Latitude span  — mean: {lat_span.mean():.2f}°, median: {lat_span.median():.2f}°, range: {lat_span.min():.2f}° – {lat_span.max():.2f}°")
print(f"Longitude span — mean: {lon_span.mean():.2f}°, median: {lon_span.median():.2f}°, range: {lon_span.min():.2f}° – {lon_span.max():.2f}°")

In [ ]:
# How large is the flooded area per event? (computed from pflood × per-patch equal-area geometry)
ext = flood_case_df['max_flood_extent_km2']
print(f"Max flood extent per case — mean: {ext.mean():.0f} km², median: {ext.median():.0f} km², range: {ext.min():.1f} – {ext.max():.0f} km²")

In [ ]:
# Where are the flood cases located and how big are their footprints?
vis = flood_case_df.copy()
vis['lat_span'] = vis['lat_max'] - vis['lat_min']
vis['lon_span'] = vis['lon_max'] - vis['lon_min']
vis['center_lon'] = (vis['lon_min'] + vis['lon_max']) / 2
vis['center_lat'] = (vis['lat_min'] + vis['lat_max']) / 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

sc = ax1.scatter(vis['center_lon'], vis['center_lat'],
                 s=80, c=vis['max_flood_extent_km2'], cmap='YlOrRd',
                 edgecolors='black', linewidth=0.4, alpha=0.8)
plt.colorbar(sc, ax=ax1, label='Max flood extent (km²)')
ax1.set_xlabel('Longitude (°)')
ax1.set_ylabel('Latitude (°)')
ax1.set_title('Geographic distribution of flood cases')
ax1.grid(alpha=0.3)

CUTOFF = 1500
ext_all = vis['max_flood_extent_km2']
in_range = ext_all[ext_all <= CUTOFF]
outliers = ext_all[ext_all > CUTOFF]

ax2.hist(in_range, bins=15, color='coral', edgecolor='black', alpha=0.7)
ax2.set_xlim(0, CUTOFF)
ax2.set_xlabel('Max flood extent (km²)')
ax2.set_ylabel('Number of cases')
ax2.set_title('Distribution of max flood extent')
ax2.grid(axis='y', alpha=0.3)

if len(outliers):
    ax2.annotate(
        '···',
        xy=(1.0, 0), xycoords='axes fraction',
        xytext=(4, -18), textcoords='offset points',
        fontsize=13, color='dimgray', annotation_clip=False,
    )
    ax2.annotate(
        f'+ {len(outliers)} cases (>{CUTOFF:,} km², max {int(outliers.max()):,} km²)',
        xy=(0.98, 0.97), xycoords='axes fraction',
        ha='right', va='top', fontsize=8, color='dimgray', style='italic',
    )

plt.tight_layout()
plt.show()

In [ ]:
# # Export the dataframe to csv
# flood_case_df.to_csv("kurosiwo_metadata_v0.csv", index=False)

 ## Summary



 KuroSiwo is **complete and production-ready** for flood mapping. All metadata needed for model equivalence extraction is available:



 | Field | Status | Source |

 |-------|--------|--------|

 | `flood_case` | ✓ | `actid` column |

 | `date_start` | ✓ | `flood_date` where `master=False` |

 | `date_end` | ✓ | `flood_date` where `master=True` |

 | `lat/lon bounding box` | ✓ | Patch geometries reprojected to WGS84 |

 | `max_flood_extent_km²` | ✓ | Computed from `pflood` × per-patch geometry area in equal-area CRS (`EPSG:6933`, GRD only) |

 | `date_of_max_flood_extent` | ✓ | Same as `date_end` (single SAR acquisition) |



 **Next steps**: download GRD batch files, use `grid_id` to locate patches, load `pflood` as labels.

 See [KuroSiwo GitHub](https://github.com/Orion-AI-Lab/KuroSiwo) for download instructions.